In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from data_process import Anekdoot
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json

**Baseline model initialization**

TF-IDF + Logistic Regression baseline for comparison with fine-tuned BERT.
Uses identical data loading, filtering, balanced sampling, and train/val split as the BERT notebook.

In [ ]:
# TF-IDF vectorizer — character n-grams (1-3) work well for Estonian morphology
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(1, 3),
    max_features=50000,
    sublinear_tf=True
)
print("TF-IDF vectorizer initialized (char n-grams 1-3, max_features=50000)")

In [ ]:
# Initialize data — identical to BERT notebook
file_path = "anekdoodid_with_types_inimene.csv"

an = Anekdoot(file_path=file_path)

dataframe = an.read_dataframe()
X_anecdotes = an.extract_main_col(dataframe, "Anekdoot").tolist()
y_anecdotes = an.extract_main_col(dataframe, "anekdoodi tüüp").tolist()

# Filter out samples with NaN labels (both actual NaN and literal "nan" string)
mask = [pd.notna(y) and str(y).strip().lower() != "nan" for y in y_anecdotes]
X_anecdotes = [x for x, m in zip(X_anecdotes, mask) if m]
y_anecdotes = [y for y, m in zip(y_anecdotes, mask) if m]

print(f"After removing NaN labels: {len(X_anecdotes)} samples")

# --- Balanced sampling: soft-cap per class ---
def balanced_sample(X, y, n_per_class=None, random_state=42):
    """
    Return a class-balanced subset of (X, y).
    n_per_class: target samples per class. Classes with fewer samples than
    n_per_class use all their available data instead of being excluded.
    If None, uses the minimum class count (hard-balanced).
    """
    import random
    rng = random.Random(random_state)

    class_indices = {}
    for idx, label in enumerate(y):
        class_indices.setdefault(label, []).append(idx)

    print("\nClass distribution (full dataset):")
    for cls, idxs in sorted(class_indices.items()):
        print(f"  {cls}: {len(idxs)}")

    if n_per_class is None:
        n_per_class = min(len(v) for v in class_indices.values())

    print(f"\nTarget {n_per_class} samples per class (soft-cap: smaller classes use all available):")
    sampled_X, sampled_y = [], []
    for label, idxs in sorted(class_indices.items()):
        take = min(n_per_class, len(idxs))
        chosen = rng.sample(idxs, take)
        print(f"  {label}: {take}")
        for i in chosen:
            sampled_X.append(X[i])
            sampled_y.append(y[i])

    total = len(sampled_X)
    print(f"\n{len(class_indices)} classes \u2192 {total} total samples")

    combined = list(zip(sampled_X, sampled_y))
    rng.shuffle(combined)
    sampled_X, sampled_y = zip(*combined)
    return list(sampled_X), list(sampled_y)

# Same n_per_class and random_state as BERT notebook for a fair comparison
X_anecdotes_subset, y_anecdotes_subset = balanced_sample(
    X_anecdotes, y_anecdotes, n_per_class=200, random_state=42
)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_anecdotes_subset)
num_labels = len(label_encoder.classes_)

print(f"\nClasses: {label_encoder.classes_}")
print(f"Number of classes: {num_labels}")
print(f"Dataset size: {len(X_anecdotes_subset)}")

In [ ]:
# Split data — identical parameters to BERT notebook (stratified, 80/20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_anecdotes_subset, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Number of classes: {num_labels}")

In [ ]:
# Fit TF-IDF on training data only, then transform both splits
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"TF-IDF feature matrix — train: {X_train_tfidf.shape}, val: {X_val_tfidf.shape}")

# Train Logistic Regression
# max_iter=1000 to ensure convergence; C=1.0 is a standard regularisation starting point
classifier = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42
)

print("\nTraining Logistic Regression...")
classifier.fit(X_train_tfidf, y_train)
print("Training complete!")

# Save model and vectorizer for reuse
import pickle
checkpoint_dir = "checkpoints_baseline"
os.makedirs(checkpoint_dir, exist_ok=True)
with open(os.path.join(checkpoint_dir, "tfidf_vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)
with open(os.path.join(checkpoint_dir, "logistic_regression.pkl"), "wb") as f:
    pickle.dump(classifier, f)
with open(os.path.join(checkpoint_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)
print(f"Saved model artifacts to {checkpoint_dir}/")

In [ ]:
# Evaluation — identical format to BERT notebook
y_pred = classifier.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.4f}")

unique_labels = sorted(set(y_val) | set(y_pred))
target_names_used = [label_encoder.classes_[i] for i in unique_labels]

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred,
    labels=unique_labels,
    target_names=target_names_used
))

In [ ]:

class_names = label_encoder.classes_

# ── 1. Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.4, ax=ax
)
ax.set_xlabel("Predicted label", fontsize=12)
ax.set_ylabel("True label", fontsize=12)
ax.set_title("Baseline (TF-IDF + LR) — Confusion Matrix (validation set)", fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("baseline_confusion_matrix.png", dpi=150)
plt.show()
print("Saved: baseline_confusion_matrix.png")

# ── 2. Per-class F1 bar chart ─────────────────────────────────────────────────
report = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
f1_scores = [report[str(i)]['f1-score'] for i in range(len(class_names))]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(class_names, f1_scores, color='darkorange', edgecolor='white')
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1-score")
ax.set_title("Baseline (TF-IDF + LR) — Per-class F1-score (validation set)")
plt.xticks(rotation=45, ha='right', fontsize=9)
ax.axhline(np.mean(f1_scores), color='firebrick', linestyle='--', linewidth=1.5, label=f'Macro avg F1 = {np.mean(f1_scores):.3f}')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("baseline_per_class_f1.png", dpi=150)
plt.show()
print("Saved: baseline_per_class_f1.png")

# ── 3. Summary metrics ────────────────────────────────────────────────────────
macro_f1    = report['macro avg']['f1-score']
weighted_f1 = report['weighted avg']['f1-score']
val_acc     = accuracy_score(y_val, y_pred)
print(f"\nBaseline Summary")
print(f"  Validation accuracy : {val_acc:.4f}")
print(f"  Macro F1            : {macro_f1:.4f}")
print(f"  Weighted F1         : {weighted_f1:.4f}")

In [ ]:

# Export baseline model for post-hoc analysis (SHAP / LIME)

export_dir = "baseline_export"
os.makedirs(export_dir, exist_ok=True)

# 1. Vectorizer
with open(os.path.join(export_dir, "tfidf_vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)
print(f"TF-IDF vectorizer saved to {export_dir}/tfidf_vectorizer.pkl")

# 2. Classifier
with open(os.path.join(export_dir, "logistic_regression.pkl"), "wb") as f:
    pickle.dump(classifier, f)
print(f"Logistic Regression saved to {export_dir}/logistic_regression.pkl")

# 3. Label encoder
with open(os.path.join(export_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder saved to {export_dir}/label_encoder.pkl")

# 4. Label map JSON — mirrors the BERT export for consistent analysis scripts
label_map = {int(i): str(cls) for i, cls in enumerate(label_encoder.classes_)}
with open(os.path.join(export_dir, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f"Label map saved to {export_dir}/label_map.json")

print("\n--- Reload example (use this in your analysis notebook) ---")
print(f"""
import pickle, json

with open("{export_dir}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
with open("{export_dir}/logistic_regression.pkl", "rb") as f:
    classifier = pickle.load(f)
with open("{export_dir}/label_map.json") as f:
    label_map = json.load(f)   # {{0: 'blondiin', 1: 'ema', ...}}

# SHAP usage:
import shap
explainer = shap.LinearExplainer(classifier, X_train_tfidf, feature_dependence="independent")
shap_values = explainer.shap_values(X_val_tfidf)
""")